# Лаборатория 3. v3: кладём в запрос весь сайт

**Что мы сделаем:** дадим боту факты самым прямым способом — положим в каждый запрос
все страницы сайта. Бот сразу начнёт отвечать правильно. А потом посмотрим, во что это
обходится, когда страниц становится тридцать, а разговор — длинным.

| Шаг | Что делаем | Кто пишет |
|---|---|---|
| 1 | Подключаемся, берём страницы сайта «Полярис» | дано |
| 2 | v3: весь сайт в запросе, замер на журнале случаев | пишем вместе |
| 3 | Сколько это стоит и сколько длится | дано |
| 4 | Что будет на 30 страницах: цена и время по размеру | дано |
| 5 | Диалог: история растёт, бот дорожает с каждым ходом | пишем вместе |
| 6 | Окно истории и пересказ | пишем вместе |
| 7 | Куда упёрлись: сводка и решение | дано |
| 8 | Задания | пиши сам |

**Запросов к модели:** около 30. Лимит учебного доступа — 120 в час, так что не
запускайте весь ноутбук несколько раз подряд. **Цена:** этот ноутбук дороже предыдущих — запросы
большие. В шаге 3 посчитаем точно.

In [ ]:
!pip -q install openai

## Шаг 1. Подключаемся и берём сайт `[дано]`

Страницы «Поляриса» лежат в репозитории курса. Здесь мы берём **уже очищенный текст**:
как превращать HTML в текст, разберём в модуле 4 — сейчас важнее сама идея.

In [ ]:
import getpass
import json
import os
import time

from openai import OpenAI


def iz_sekretov(imya, po_umolchaniyu=None):
    try:
        from google.colab import userdata
        znachenie = userdata.get(imya)
        if znachenie:
            return znachenie
    except Exception:
        pass
    return os.environ.get(imya) or po_umolchaniyu


BASE_URL = iz_sekretov("AI_BASE_URL", "https://ai9.adelfos.ru/api/v1")
MODEL = iz_sekretov("AI_MODEL", "qwen/qwen3.7-flash")
API_KEY = iz_sekretov("AI_KEY")

client = None
while client is None:
    if not API_KEY:
        API_KEY = getpass.getpass("Ключ или код доступа: ")
    probnyy = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60, max_retries=0)
    try:
        probnyy.models.list()
        client = probnyy
        print(f"Подключились. Адрес: {BASE_URL}, модель: {MODEL}")
    except Exception as oshibka:
        print(f"Не подошло: {type(oshibka).__name__} — {str(oshibka)[:120]}")
        API_KEY = None

CENA_VHODA, CENA_VYHODA = 0.03, 0.13   # $ за миллион токенов; под свою модель поправьте

In [ ]:
STRANICY = {
    "Главная": """Сервисный центр «Полярис» чинит стиральные машины, посудомойки, холодильники
и кофемашины с 2014 года. Работаем с частными клиентами и с организациями по договору.
Как мы работаем: вы оставляете заявку на сайте или звоните; мастер приезжает и проводит
диагностику — 1200 рублей, при согласии на ремонт диагностика бесплатная; стоимость
согласуется до начала работ; после ремонта выдаём акт и гарантийный талон.
Мастерская на Заводской, 15 открыта с понедельника по пятницу с 9:00 до 20:00,
в субботу с 10:00 до 17:00. Воскресенье — выходной. Заявки на сайте принимаются круглосуточно.""",

    "Услуги и цены": """Цены указаны без стоимости запчастей, окончательную сумму мастер называет
после диагностики. Стиральные машины: диагностика на дому 1200 ₽ (бесплатно при согласии
на ремонт); замена подшипников от 4900 ₽, срок 1–2 дня; замена нагревательного элемента
от 2200 ₽, обычно в день обращения; чистка от накипи и засоров 1800 ₽.
Посудомоечные машины: замена насоса от 3400 ₽; ремонт платы управления от 5600 ₽,
срок до 5 рабочих дней. Холодильники: заправка хладагентом от 3900 ₽; замена компрессора
от 8700 ₽, срок 2–4 дня. Кофемашины: обслуживание заварочного блока 2400 ₽; ремонт помпы
от 3100 ₽. Срочный выезд в день обращения — плюс 900 ₽ к стоимости работ.""",

    "Гарантия": """На выполненные работы гарантия 12 месяцев, на установленные нами запчасти —
6 месяцев. Срок считается со дня подписания акта. Гарантия покрывает повторную поломку
того же узла и дефект нашей запчасти: повторный ремонт бесплатный, включая выезд мастера.
Гарантия не действует, если технику вскрывали в другом сервисе; если повреждение вызвано
попаданием воды, скачком напряжения или ударом; если нарушены правила эксплуатации;
если утерян акт и его нет в нашей базе. Для обращения назовите номер акта, мастер
приезжает в течение двух рабочих дней.""",

    "Приём техники": """Стиральные машины, посудомойки и холодильники мастер чинит на дому.
Выезд в пределах города бесплатный, за пределами — 40 ₽ за километр. Кофемашины,
микроволновки и мелкую технику привозят на Заводскую, 15; можно вызвать курьера:
700 ₽ в одну сторону, 1200 ₽ туда и обратно. Диагностика в мастерской занимает до двух
рабочих дней, ремонт — от одного дня до недели в зависимости от наличия запчасти.
Первые 14 дней хранение бесплатное, дальше 100 ₽ в день; технику, не забранную
в течение 60 дней, вправе передать на утилизацию по договору.""",

    "Контакты": """Мастерская: Заводская, 15, вход со двора. Телефон +7 (999) 000-00-00,
почта help@polaris-service.example. Приём заявок по телефону с 9:00 до 20:00 в будни,
заявка на сайте принимается круглосуточно. Для организаций работаем по договору оказания
услуг с отсрочкой платежа 10 рабочих дней, закрывающие документы отправляем в течение
5 рабочих дней после ремонта.""",
}

VES_SAYT = "\n\n".join(f"=== {imya} ===\n{tekst}" for imya, tekst in STRANICY.items())
print(f"Страниц: {len(STRANICY)}, символов: {len(VES_SAYT)}")

SLUCHAI = [
    {"id": "subbota", "vopros": "Во сколько вы закрываетесь в субботу?", "zhdem": "17:00"},
    {"id": "garantiya", "vopros": "Какая у вас гарантия на ремонт?", "zhdem": "12 месяц"},
    {"id": "podshipniki", "vopros": "Сколько стоит замена подшипников в стиральной машине?", "zhdem": "4900"},
    {"id": "samokaty", "vopros": "Вы чините электросамокаты?", "zhdem": "не знаю"},
    {"id": "diagnostika", "vopros": "Диагностика платная?", "zhdem": "1200"},
]

## Шаг 2. v3: весь сайт в запросе `[пишем вместе]`

Идея простая до неприличия: положить весь текст сайта в правила и потребовать отвечать
только по нему. Это называется **грунтованием** — ответ «заземляется» на данные,
которые мы дали сами.

In [ ]:
PRAVILA_V3 = """Ты помощник сервисного центра «Полярис».
Отвечай ТОЛЬКО по тексту из блока ДАННЫЕ САЙТА, который идёт ниже.
Если ответа там нет — ответь ровно: «Не знаю, уточните у оператора».
Отвечай кратко, одно-два предложения.

ДАННЫЕ САЙТА:
""" + VES_SAYT


def sprosit(soobshcheniya, max_tokens=150):
    nachalo = time.perf_counter()
    otvet = client.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=max_tokens, messages=soobshcheniya)
    sekundy = time.perf_counter() - nachalo
    usage = otvet.usage
    stoimost = (usage.prompt_tokens * CENA_VHODA + usage.completion_tokens * CENA_VYHODA) / 1e6
    return (otvet.choices[0].message.content or "").strip(), usage, sekundy, stoimost


def bot_v3(vopros):
    tekst, *_ = sprosit([{"role": "system", "content": PRAVILA_V3},
                         {"role": "user", "content": vopros}])
    return tekst


def proverit(otvet, zhdem):
    nizhniy = otvet.lower()
    if zhdem == "не знаю":
        return any(s in nizhniy for s in ("не зна", "уточните", "нет информац"))
    return zhdem.lower() in nizhniy


def progon(bot, nazvanie):
    print(f"=== {nazvanie} ===")
    verno = 0
    for sluchay in SLUCHAI:
        otvet = bot(sluchay["vopros"])
        ok = proverit(otvet, sluchay["zhdem"])
        verno += ok
        print(f"{'✅' if ok else '❌'} {sluchay['id']:<12} ждём «{sluchay['zhdem']}» | {otvet[:70]}")
    print(f"Верных ответов: {verno} из {len(SLUCHAI)}\n")
    return verno


verno_v3 = progon(bot_v3, "v3: весь сайт в запросе")

**Что посмотреть в выводе:**

1. При подготовке лаборатории вышло **5 из 5** — против 1 из 5 в модулях 1 и 2.
   Одно изменение: у бота появились факты.
2. Посмотрите на случай `samokaty`: теперь отказ не выдуманный, а обоснованный —
   в данных сайта такой услуги нет.
3. Это и есть **грунтование**: не просить модель «не выдумывать», а дать ей то,
   из чего можно ответить.

## Шаг 3. Сколько это стоит `[дано]`

За удобство мы платим: в каждый запрос уезжает весь сайт.

In [ ]:
tekst, usage, sekundy, stoimost = sprosit([{"role": "system", "content": PRAVILA_V3},
                                           {"role": "user", "content": "Диагностика платная?"}])
print(f"Ответ: {tekst}")
print(f"Вход: {usage.prompt_tokens} токенов, выход: {usage.completion_tokens}")
print(f"Время: {sekundy:.1f} с, цена одного вопроса: ${stoimost:.6f}")
print(f"\nИз них на сам вопрос приходится примерно {len('Диагностика платная?') // 3} токенов,")
print("всё остальное — сайт, который едет заново на каждом вопросе.")
for v in (1_000, 10_000, 100_000):
    print(f"   {v:>7} вопросов: ${stoimost * v:>8.2f}")

**Что посмотреть в выводе:** при подготовке лаборатории вход составил 955 токенов,
из которых на сам вопрос пришлось около шести. Мы платим почти целиком за пересылку
одного и того же текста — и так на **каждом** вопросе.

## Шаг 4. А если страниц тридцать? `[дано]`

У настоящего сайта не пять страниц. Смоделируем рост: будем повторять наш сайт
несколько раз и смотреть, как меняются токены, время и цена. Ответ при этом должен
остаться верным — проверим и его.

In [ ]:
VOPROS_PROVERKI = "Сколько стоит замена подшипников в стиральной машине?"
print(f"{'страниц':>8} {'символов':>9} {'вход':>7} {'время':>7} {'цена':>10} {'ответ верный':>13}")
izmereniya = []
for mnozhitel in (1, 2, 4, 6):
    bolshoy_sayt = "\n\n".join([VES_SAYT] * mnozhitel)
    pravila = PRAVILA_V3.split("ДАННЫЕ САЙТА:")[0] + "ДАННЫЕ САЙТА:\n" + bolshoy_sayt
    tekst, usage, sekundy, stoimost = sprosit([{"role": "system", "content": pravila},
                                               {"role": "user", "content": VOPROS_PROVERKI}])
    ok = "4900" in tekst
    izmereniya.append((len(STRANICY) * mnozhitel, usage.prompt_tokens, sekundy, stoimost, ok))
    print(f"{len(STRANICY) * mnozhitel:>8} {len(bolshoy_sayt):>9} {usage.prompt_tokens:>7} "
          f"{sekundy:>6.1f}с {stoimost:>10.6f} {'✅' if ok else '❌':>13}")

pervoe, poslednee = izmereniya[0], izmereniya[-1]
print(f"\nОт {pervoe[0]} до {poslednee[0]} страниц: токенов в {poslednee[1] / pervoe[1]:.1f} раза больше, "
      f"время в {poslednee[2] / pervoe[2]:.1f} раза, цена в {poslednee[3] / pervoe[3]:.1f} раза.")

**Что посмотреть в выводе:**

1. Токены и цена растут **прямо пропорционально** размеру сайта: при подготовке
   лаборатории 5 страниц дали 961 токен, 30 страниц — 5246, то есть в 5,5 раза больше.
2. Время почти не изменилось: на таких размерах читает модель быстро. Оно станет
   заметным на сотнях страниц — а вот цена растёт сразу.
3. Ответ пока остаётся верным. То есть проблема не в качестве, а в **цене и скорости**.
   Именно поэтому её легко не заметить на маленьком сайте и очень заметно получить
   в счёте на большом.

И есть предел, которого мы здесь не достигли: у модели ограничено контекстное окно —
сколько текста она может принять за раз. Сайт на тысячу страниц в запрос просто не влезет.

## Шаг 5. Диалог: вторая беда `[пишем вместе]`

Гость редко задаёт один вопрос. Он спрашивает, уточняет, потом ещё раз уточняет —
и бот должен помнить, о чём речь.

Модель между запросами не помнит **ничего**. Чтобы поддержать разговор, программа
пересылает всю историю заново. Посмотрим, во что это обходится.

In [ ]:
RAZGOVOR = [
    "Здравствуйте! У меня шумит стиральная машина при отжиме.",
    "Сколько будет стоить починить?",
    "А сколько времени займёт?",
    "Хорошо. А гарантия на такой ремонт есть?",
    "Мастер приедет ко мне или нужно везти машину?",
    "Напомните, о какой поломке я спрашивал в начале?",
]

istoriya = [{"role": "system", "content": PRAVILA_V3}]
vsego_vhod, vsego_cena = 0, 0.0
print(f"{'ход':>3} {'сообщений':>10} {'вход':>7} {'цена хода':>11}  ответ")
for nomer, vopros in enumerate(RAZGOVOR, 1):
    istoriya.append({"role": "user", "content": vopros})
    tekst, usage, sekundy, stoimost = sprosit(istoriya)
    istoriya.append({"role": "assistant", "content": tekst})
    vsego_vhod += usage.prompt_tokens
    vsego_cena += stoimost
    print(f"{nomer:>3} {len(istoriya) - 1:>10} {usage.prompt_tokens:>7} {stoimost:>11.6f}  {tekst[:60]}")

print(f"\nЗа разговор из {len(RAZGOVOR)} ходов: {vsego_vhod} токенов входа, ${vsego_cena:.6f}")
print(f"На 1000 таких разговоров: ${vsego_cena * 1000:.2f}")

**Что посмотреть в выводе:**

1. Колонка «вход» растёт с каждым ходом: к сайту добавляется вся предыдущая переписка.
2. Последний вопрос — проверка памяти. Бот должен вспомнить про шум при отжиме:
   вся история у него перед глазами.
3. Цена разговора из шести ходов уже заметно больше, чем шесть отдельных вопросов.
   И чем дольше разговор, тем хуже: каждый ход заново оплачивает все предыдущие.

## Шаг 6. Окно и пересказ `[пишем вместе]`

Самое простое лечение — отправлять не всю историю, а последние несколько сообщений.
Это **скользящее окно**. Посмотрим, что оно экономит и что ломает.

In [ ]:
def razgovor_s_oknom(okno=None, pereskaz=False):
    """okno=None — вся история. okno=N — последние N сообщений (+ пересказ, если включён)."""
    istoriya, itog_vhod, itog_cena, kratko = [], 0, 0.0, ""
    for vopros in RAZGOVOR:
        istoriya.append({"role": "user", "content": vopros})
        vidimaya = istoriya if okno is None else istoriya[-okno:]
        zapros = [{"role": "system", "content": PRAVILA_V3}]
        if pereskaz and kratko:
            zapros.append({"role": "system", "content": "Кратко о начале разговора: " + kratko})
        zapros += vidimaya
        tekst, usage, _, stoimost = sprosit(zapros)
        istoriya.append({"role": "assistant", "content": tekst})
        itog_vhod += usage.prompt_tokens
        itog_cena += stoimost
        if pereskaz and len(istoriya) > okno:
            vypavshie = istoriya[:-okno]
            kratko, _, _, cena_pereskaza = sprosit(
                [{"role": "system", "content": "Перескажи разговор в одном предложении. "
                                               "Обязательно сохрани, какая техника и какая поломка."},
                 {"role": "user", "content": "\n".join(s["content"][:200] for s in vypavshie)}],
                max_tokens=80)
            itog_cena += cena_pereskaza
    return itog_vhod, itog_cena, tekst


def pomnit_nachalo(otvet):
    return any(s in otvet.lower() for s in ("отжим", "шум", "стиральн"))


# «Вся история» уже посчитана в шаге 5 — не тратим на неё запросы заново.
print(f"{'вся история':<20} вход {vsego_vhod:>6} токенов, ${vsego_cena:.6f}, "
      f"помнит начало: {'✅' if pomnit_nachalo(tekst) else '❌'}")
for nazvanie, parametry in (("окно 4 сообщения", {"okno": 4}),
                            ("окно 4 + пересказ", {"okno": 4, "pereskaz": True})):
    vhod, cena_razgovora, posledniy = razgovor_s_oknom(**parametry)
    print(f"{nazvanie:<20} вход {vhod:>6} токенов, ${cena_razgovora:.6f}, "
          f"помнит начало: {'✅' if pomnit_nachalo(posledniy) else '❌'}")
    print(f"{'':<20} последний ответ: {posledniy[:80]}")

**Что посмотреть в выводе:**

1. **Окно** экономит на истории, но выкидывает начало разговора: на вопрос «о какой
   поломке я спрашивал» бот с маленьким окном ответить не может.
2. **Пересказ** возвращает память: старые сообщения сжимаются в одно предложение.
   Но он стоит **дополнительного запроса** — это видно в цене.
3. И заметьте: экономия на истории — копейки на фоне сайта, который едет в каждом
   запросе целиком. Мы лечим не ту болезнь.

## Шаг 7. Куда мы упёрлись `[дано]`

Соберём итог модуля в одну таблицу.

In [ ]:
print(f"{'версия':<28} {'верных':>7} {'вход на вопрос':>15}")
print(f"{'v1: просто спрашиваем':<28} {'1 из 5':>7} {'~40':>15}")
print(f"{'v2: + надёжность и схема':<28} {'1 из 5':>7} {'~60':>15}")
print(f"{'v3: весь сайт в запросе':<28} {str(verno_v3) + ' из 5':>7} {usage.prompt_tokens:>15}")
print("""
Что хорошо: факты появились, бот перестал выдумывать, отказ стал обоснованным.
Что плохо:  каждый вопрос платит за весь сайт; на большом сайте это дорого,
            медленно, а однажды текст просто не влезет в контекстное окно.
Что дальше: отправлять не весь сайт, а только те куски, которые относятся
            к вопросу. Это и есть RAG — модуль 4.""")

## Шаг 8. Задания `[пиши сам]`

1. **Найдите предел.** Увеличивайте множитель в шаге 4 (8, 16, 32…), пока сервер
   не откажется принимать запрос. На каком размере это произошло и с каким кодом ошибки?
2. **Потерянная середина.** Спрячьте в середину большого сайта строку «Пароль от вайфая
   для гостей: polaris2026» и спросите о нём при множителе 1 и 6. Находит ли бот факт
   в обоих случаях?
3. **Цена честного отказа.** Сколько стоит ответ «Не знаю, уточните у оператора»?
   Подсказка: вход тот же самый. Что это говорит о вопросах, на которые бот заведомо
   не сможет ответить?
4. **Окно поменьше.** Поставьте `okno=2` и прогоните разговор. С какого хода бот начинает
   терять нить? Сравните с `okno=8`.

## Что записать в файлы курса

**`decisions.md`:**

> **Что сравнивали.** v2 (без документов) и v3 (весь сайт в каждом запросе) на 5 случаях;
> плюс замер цены при росте сайта в 1, 2, 4 и 6 раз.
> **Числа.** Верных ответов: было 1 из 5, стало столько-то. Вход на вопрос: было ~60
> токенов, стало столько-то. Цена растёт пропорционально размеру сайта.
> **Что выбрали.** v3 как рабочую версию на ближайший модуль: факты важнее цены.
> **Чем пожертвовали.** Ценой и скоростью; на большом сайте решение нежизнеспособно.
> **Когда пересмотреть.** Как только страниц станет больше десяти или вопросов —
> больше тысячи в месяц.

**`cases.jsonl`** — добавьте случай из задания 2 про пароль вайфая, если бот его потерял.

## Что унести с собой

* **Грунтование** — единственный честный способ дать модели факты: положить их в запрос.
* Правильный ответ появляется не от просьб, а от данных.
* Цена запроса растёт вместе с объёмом данных, и платится она на **каждом** вопросе.
* Диалог дорожает с каждым ходом: история пересылается заново.
* Окно и пересказ лечат историю, но не лечат главного — пересылки всего сайта.